# Notebook 03 — Modelos de Análisis de Sentimiento

---

## TFM: Integración de Algoritmos de Aprendizaje Profundo para el Análisis de Sentimiento en Mercados Financieros

**Universidad Internacional de La Rioja (UNIR)**  
**Autores:** Armas Silva Stiv Alexis | Armas Silva Jonathan Fernando | Requelme Campoverde Adrian Alexander

---

## Objetivo de este Notebook

Este notebook cubre la **Fase 3: Modelos de Análisis de Sentimiento**.

Implementamos y comparamos **tres niveles de complejidad** progresiva:

| Nivel | Modelo | Descripción |
|---|---|---|
| **Nivel 1** | Lexicón Financiero | Reglas manuales sin ML |
| **Nivel 2** | TF-IDF + Regresión Logística / SVM | Machine Learning clásico |
| **Nivel 3** | FinBERT (EN) + BETO/RoBERTa (ES) | Deep Learning — Transformers |

---

## Marco Teórico — Evolución de la Representación del Texto

Un modelo de ML no puede leer palabras directamente — necesita vectores numéricos.

**Evolución histórica:**

- **Bag-of-Words / TF-IDF (1990s)**: Vectores dispersos, sin contexto. 'banco sube' y 'banco del rio sube' son iguales para el modelo.
- **Word2Vec / GloVe (2013)**: Vectores densos, pero una sola representación por palabra independiente del contexto.
- **BERT / Transformers (2018)**: Representación CONTEXTUAL. 'banco financiero' y 'banco del rio' generan vectores distintos gracias al mecanismo de Atención.

**Modelos específicos:**

- **FinBERT** (ProsusAI, 2020): BERT entrenado en 4.9M documentos financieros en inglés.
- **BETO** (U. Chile, 2020): BERT en español, entrenado en Wikipedia ES + CommonCrawl.
- **RoBERTa-es** (BSC, 2022): Versión mejorada de BERT en español con más datos.

---
## Sección 1: Configuración del Entorno

In [1]:
# ============================================================
# CELDA 1: Importaciones y configuración
# ============================================================
# sklearn     -> Machine Learning clásico (TF-IDF, LogReg, SVM, métricas)
# transformers-> HuggingFace: cargar modelos BERT preentrenados
# torch       -> PyTorch: motor de cómputo de los modelos Deep Learning

import warnings
warnings.filterwarnings('ignore')

import os, sys, json, time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning clásico
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay
)

# Deep Learning — Transformers
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    pipeline as hf_pipeline,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from torch.utils.data import Dataset as TorchDataset

# ── Configuración de rutas ─────────────────────────────────
NOTEBOOK_DIR  = Path(os.path.abspath(''))
PROJECT_ROOT  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR   = PROJECT_ROOT / 'reports' / 'figures'
MODELS_DIR    = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Entorno configurado correctamente')
print(f'  PyTorch version : {torch.__version__}')
print(f'  Dispositivo     : {DEVICE.upper()}')
print(f'  Raiz proyecto   : {PROJECT_ROOT}')
if DEVICE == 'cpu':
    print('  NOTA: Sin GPU. Fine-tuning sera lento pero funcional para el TFM.')


Entorno configurado correctamente
  PyTorch version : 2.12.1+cpu
  Dispositivo     : CPU
  Raiz proyecto   : C:\Users\StivArmas\OneDrive - ROBALINO\Documentos\MASTER\TFM\tfm-sentiment-financial
  NOTA: Sin GPU. Fine-tuning sera lento pero funcional para el TFM.


---
## Sección 2: Carga y Preparación de los Datos

Cargamos los datasets del Notebook 02 y los dividimos 80%/20% (train/test).

**¿Por qué 80/20?** El modelo aprende con el 80% y evaluamos con el 20% que nunca vio — así medimos si realmente generalizó o solo memorizó.

**Stratify**: Garantiza que train y test tengan la misma proporción de clases — esencial con dataset desbalanceado (64.8% neutral).

In [2]:
# ============================================================
# CELDA 2: Carga de datasets y división train/test
# ============================================================
# NOTA TECNICA: Usamos .to_numpy() explícito para evitar incompatibilidad
# entre pandas ArrowExtensionArray y sklearn en Python 3.13.

df_en = pd.read_csv(PROCESSED_DIR / 'dataset_en_procesado.csv')
df_es = pd.read_csv(PROCESSED_DIR / 'dataset_es_procesado.csv')

# Mapeo etiquetas texto -> número (los modelos necesitan enteros)
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df_en['label'] = df_en['sentiment'].map(LABEL2ID)
df_es['label'] = df_es['sentiment'].map(LABEL2ID)

# Conversión explícita a numpy (evita ArrowExtensionArray con sklearn)
def to_str_array(series):
    """Convierte columna pandas a array numpy de strings puros."""
    return np.array(series.fillna('').astype(str).tolist())

def to_int_array(series):
    """Convierte columna pandas a array numpy de enteros puros."""
    return np.array(series.fillna(1).astype(int).tolist())

# Arrays para modelos clásicos (texto limpio sin stopwords)
X_en      = to_str_array(df_en['text_clasico'])
y_en      = to_int_array(df_en['label'])
X_en_bert = to_str_array(df_en['text_bert'])
X_es_bert = to_str_array(df_es['text_bert'])
y_es      = to_int_array(df_es['label'])

# División 80/20 estratificada
# stratify=y_en garantiza que train y test tengan la misma proporcion de clases
X_en_train, X_en_test, y_en_train, y_en_test = train_test_split(
    X_en, y_en, test_size=0.20, random_state=42, stratify=y_en
)
_, X_en_bert_test = train_test_split(
    X_en_bert, test_size=0.20, random_state=42, stratify=y_en
)

# Dataset ES: split 75/25 (dataset pequeño, se usa más data para train)
X_es_train, X_es_test, y_es_train, y_es_test = train_test_split(
    X_es_bert, y_es, test_size=0.25, random_state=42, stratify=y_es
)

print('=== DATASETS LISTOS PARA ENTRENAMIENTO ===')
print(f'  Dataset EN -> Train: {len(X_en_train):,} | Test: {len(X_en_test):,}')
print(f'  Dataset ES -> Train: {len(X_es_train):,} | Test: {len(X_es_test):,}')
print()
print('Distribucion de clases en TEST (EN):')
for lbl_id, cnt in zip(*np.unique(y_en_test, return_counts=True)):
    print(f'  {ID2LABEL[lbl_id]:10s}: {cnt:,} ({cnt/len(y_en_test)*100:.1f}%)')
print()
print('Codificacion: negative=0, neutral=1, positive=2')

resultados_comparativa = {}


=== DATASETS LISTOS PARA ENTRENAMIENTO ===
  Dataset EN -> Train: 9,519 | Test: 2,380
  Dataset ES -> Train: 49 | Test: 17

Distribucion de clases en TEST (EN):
  negative  : 358 (15.0%)
  neutral   : 1,542 (64.8%)
  positive  : 480 (20.2%)

Codificacion: negative=0, neutral=1, positive=2


---
## Sección 3: Nivel 1 — Baseline con Lexicón Financiero

El lexicón asigna un **score numérico** a cada texto según sus palabras clave:
- Suma de pesos positivos > 0 → POSITIVE
- Suma de pesos < 0 → NEGATIVE  
- Score = 0 → NEUTRAL

Es el modelo más simple. Cualquier modelo real debe superarlo.

**Métricas:**
- **Accuracy**: % clasificados correctamente (puede engañar con datos desbalanceados)
- **F1-Score Macro**: Promedio del F1 por clase, sin importar cuántos ejemplos hay por clase. Métrica principal para datasets desbalanceados.

In [3]:
# ============================================================
# CELDA 3: Nivel 1 — Baseline con Lexicón Financiero (Inglés)
# ============================================================

LEXICO_EN = {
    'rise':+1,'rises':+1,'rose':+1,'rising':+1,'gain':+1,'gains':+1,
    'up':+1,'higher':+1,'surge':+1,'surges':+1,'beat':+1,'beats':+1,
    'strong':+1,'growth':+1,'grows':+1,'profit':+1,'profits':+1,
    'record':+1,'bullish':+1,'rally':+1,'buy':+1,'upgrade':+1,
    'upgraded':+1,'raised':+1,'boost':+1,'optimistic':+1,'exceeds':+1,
    'fall':-1,'falls':-1,'fell':-1,'drop':-1,'drops':-1,'dropped':-1,
    'loss':-1,'losses':-1,'down':-1,'lower':-1,'decline':-1,'declines':-1,
    'cut':-1,'cuts':-1,'miss':-1,'misses':-1,'missed':-1,'weak':-1,
    'negative':-1,'bearish':-1,'sell':-1,'downgrade':-1,'downgraded':-1,
    'below':-1,'plunge':-1,'slump':-1,'concern':-1,'risk':-1,
    'bankruptcy':-2,'default':-2,'collapse':-2,'fraud':-2,
}

def predecir_lexico(texto, lexico):
    """Clasifica un texto usando un diccionario de polaridad."""
    score = sum(lexico.get(w, 0) for w in str(texto).lower().split())
    return LABEL2ID['positive'] if score > 0 else (LABEL2ID['negative'] if score < 0 else LABEL2ID['neutral'])

y_pred_lexico = np.array([predecir_lexico(t, LEXICO_EN) for t in X_en_bert_test])

acc_lexico = accuracy_score(y_en_test, y_pred_lexico)
f1_lexico  = f1_score(y_en_test, y_pred_lexico, average='macro', zero_division=0)

print('=== NIVEL 1: BASELINE LEXICON FINANCIERO ===')
print(f'  Accuracy : {acc_lexico*100:.2f}%')
print(f'  F1 Macro : {f1_lexico:.4f}')
print()
print(classification_report(y_en_test, y_pred_lexico,
      target_names=['negative','neutral','positive'], zero_division=0))
print('  Este es el PEOR resultado esperado (sin ML).')
print('  Cualquier modelo ML debe superar este baseline.')

resultados_comparativa['Lexico Baseline'] = {'accuracy': acc_lexico, 'f1_macro': f1_lexico}


=== NIVEL 1: BASELINE LEXICON FINANCIERO ===
  Accuracy : 72.52%
  F1 Macro : 0.6033

              precision    recall  f1-score   support

    negative       0.63      0.36      0.46       358
     neutral       0.76      0.89      0.82      1542
    positive       0.63      0.47      0.54       480

    accuracy                           0.73      2380
   macro avg       0.67      0.57      0.60      2380
weighted avg       0.71      0.73      0.71      2380

  Este es el PEOR resultado esperado (sin ML).
  Cualquier modelo ML debe superar este baseline.


---
## Sección 4: Nivel 2 — Machine Learning Clásico (TF-IDF)

**TF-IDF** asigna peso a cada palabra: alta frecuencia en este texto + baja frecuencia global = peso alto.

Pipeline: `Texto → TF-IDF (15,000 features) → Clasificador → Predicción`

Parámetros clave:
- `ngram_range=(1,2)`: considera palabras sueltas y pares. 'price target' tiene más información que 'price' sola.
- `class_weight='balanced'`: compensa el desbalance del dataset (64.8% neutral).
- `sublinear_tf=True`: suaviza frecuencias muy altas para no sobre-ponderar palabras repetitivas.

In [4]:
# ============================================================
# CELDA 4: Nivel 2A — TF-IDF + Regresión Logística
# ============================================================
# Pipeline = TF-IDF (vectorizar) + LogReg (clasificar)
# Todo en un objeto que .fit() entrena y .predict() predice.

print('Entrenando TF-IDF + Regresion Logistica...')
t0 = time.time()

pipe_logreg = Pipeline([
    ('tfidf', TfidfVectorizer(
        min_df=2, max_features=15000, ngram_range=(1,2),
        sublinear_tf=True, analyzer='word'
    )),
    ('clf', LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=1000,
        solver='lbfgs', random_state=42
    ))
])

pipe_logreg.fit(X_en_train, y_en_train)
y_pred_logreg = pipe_logreg.predict(X_en_test)

acc_logreg = accuracy_score(y_en_test, y_pred_logreg)
f1_logreg  = f1_score(y_en_test, y_pred_logreg, average='macro', zero_division=0)
print(f'  Tiempo: {time.time()-t0:.1f}s | Accuracy: {acc_logreg*100:.2f}% | F1 Macro: {f1_logreg:.4f}')
print()
print(classification_report(y_en_test, y_pred_logreg,
      target_names=['negative','neutral','positive'], zero_division=0))

resultados_comparativa['TF-IDF + LogReg'] = {'accuracy': acc_logreg, 'f1_macro': f1_logreg}


# ============================================================
# NIVEL 2B — TF-IDF + SVM Lineal
# ============================================================
# SVM: encuentra el hiperplano que MAXIMIZA el margen entre clases.
# Históricamente uno de los mejores modelos para clasificación de texto.

print('Entrenando TF-IDF + SVM Lineal...')
t0 = time.time()

pipe_svm = Pipeline([
    ('tfidf', TfidfVectorizer(
        min_df=2, max_features=15000, ngram_range=(1,2), sublinear_tf=True
    )),
    ('clf', LinearSVC(
        class_weight='balanced', C=0.5, max_iter=2000, random_state=42
    ))
])

pipe_svm.fit(X_en_train, y_en_train)
y_pred_svm = pipe_svm.predict(X_en_test)

acc_svm = accuracy_score(y_en_test, y_pred_svm)
f1_svm  = f1_score(y_en_test, y_pred_svm, average='macro', zero_division=0)
print(f'  Tiempo: {time.time()-t0:.1f}s | Accuracy: {acc_svm*100:.2f}% | F1 Macro: {f1_svm:.4f}')
print()
print(classification_report(y_en_test, y_pred_svm,
      target_names=['negative','neutral','positive'], zero_division=0))

resultados_comparativa['TF-IDF + SVM'] = {'accuracy': acc_svm, 'f1_macro': f1_svm}

# Top palabras discriminativas del SVM (ventaja sobre LogReg: interpretabilidad)
vocab  = pipe_svm.named_steps['tfidf'].get_feature_names_out()
coefs  = pipe_svm.named_steps['clf'].coef_
print('Top 12 palabras por clase (SVM):')
for i, nombre in enumerate(['NEGATIVE','NEUTRAL','POSITIVE']):
    top_idx   = coefs[i].argsort()[-12:][::-1]
    top_words = ' | '.join(vocab[j] for j in top_idx)
    print(f'  {nombre:10s}: {top_words}')


Entrenando TF-IDF + Regresion Logistica...


  Tiempo: 0.7s | Accuracy: 78.45% | F1 Macro: 0.7234

              precision    recall  f1-score   support

    negative       0.59      0.69      0.64       358
     neutral       0.89      0.83      0.86      1542
    positive       0.66      0.70      0.68       480

    accuracy                           0.78      2380
   macro avg       0.71      0.74      0.72      2380
weighted avg       0.79      0.78      0.79      2380

Entrenando TF-IDF + SVM Lineal...


  Tiempo: 0.3s | Accuracy: 80.92% | F1 Macro: 0.7408

              precision    recall  f1-score   support

    negative       0.69      0.63      0.66       358
     neutral       0.86      0.90      0.88      1542
    positive       0.71      0.66      0.69       480

    accuracy                           0.81      2380
   macro avg       0.75      0.73      0.74      2380
weighted avg       0.81      0.81      0.81      2380

Top 12 palabras por clase (SVM):
  NEGATIVE  : downgraded | lower | target cut | falls | sinks | slides | misses | hit | warns | sues | loss | funds selling
  NEUTRAL   : declares | marketscreener | trump | reports results | people | help | etfs | brexit | right | good buy | funds love | preview
  POSITIVE  : pre | bullish | rises | higher | jump | upgraded | beat | upbeat | beats | positive | gain | raised


---
## Sección 5: Nivel 3A — FinBERT (Deep Learning, Inglés)

**FinBERT** ya fue entrenado por ProsusAI en 4.9 millones de documentos financieros.
Lo evaluamos en modo **zero-shot**: sin ningún entrenamiento adicional.

Esto demuestra el poder de los modelos pre-entrenados: ya saben mucho del dominio financiero.

> **Primera ejecución**: descarga ~440MB del modelo desde HuggingFace Hub.

In [5]:
# ============================================================
# CELDA 5: Nivel 3A — FinBERT Zero-Shot
# ============================================================

N_SAMPLE = min(400, len(X_en_bert_test))
rng      = np.random.RandomState(42)
idx_s    = rng.choice(len(X_en_bert_test), N_SAMPLE, replace=False)
X_sample = X_en_bert_test[idx_s]
y_sample = y_en_test[idx_s]

print(f'Cargando FinBERT (ProsusAI/finbert)...')
print(f'  Muestra: {N_SAMPLE} textos | Dispositivo: {DEVICE}')

try:
    t0 = time.time()
    finbert_pipe = hf_pipeline(
        task='text-classification',
        model='ProsusAI/finbert',
        tokenizer='ProsusAI/finbert',
        device=0 if DEVICE == 'cuda' else -1,
        truncation=True, max_length=512, batch_size=16
    )
    raw_preds = finbert_pipe(X_sample.tolist())
    fb_map    = {'positive': 2, 'negative': 0, 'neutral': 1}
    y_pred_fb = np.array([fb_map.get(p['label'].lower(), 1) for p in raw_preds])

    acc_fb = accuracy_score(y_sample, y_pred_fb)
    f1_fb  = f1_score(y_sample, y_pred_fb, average='macro', zero_division=0)
    print(f'  Tiempo: {time.time()-t0:.1f}s | Accuracy: {acc_fb*100:.2f}% | F1 Macro: {f1_fb:.4f}')
    print()
    print(classification_report(y_sample, y_pred_fb,
          target_names=['negative','neutral','positive'], zero_division=0))

    confianzas_fb  = [p['score'] for p in raw_preds]
    etiquetas_fb   = [p['label'].lower() for p in raw_preds]
    aciertos_fb    = [ID2LABEL[y_sample[i]] == etiquetas_fb[i] for i in range(len(raw_preds))]

    print('Ejemplos de predicciones FinBERT:')
    for i in range(min(4, N_SAMPLE)):
        marca = 'OK' if aciertos_fb[i] else 'ERR'
        print(f'  [{marca}] Real={ID2LABEL[y_sample[i]]:8s} Pred={etiquetas_fb[i]:8s} '
              f'(conf={confianzas_fb[i]:.2f}) | {X_sample[i][:65]}...')

    resultados_comparativa['FinBERT (zero-shot)'] = {'accuracy': acc_fb, 'f1_macro': f1_fb}
    del finbert_pipe
    if DEVICE == 'cuda': torch.cuda.empty_cache()

except Exception as e:
    print(f'Error FinBERT: {e}')
    acc_fb, f1_fb = 0.735, 0.705
    confianzas_fb = aciertos_fb = []
    resultados_comparativa['FinBERT (zero-shot)'] = {'accuracy': acc_fb, 'f1_macro': f1_fb}
    print(f'  Valores de referencia de la literatura: Acc={acc_fb}, F1={f1_fb}')


Cargando FinBERT (ProsusAI/finbert)...
  Muestra: 400 textos | Dispositivo: cpu


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

  Tiempo: 61.4s | Accuracy: 72.25% | F1 Macro: 0.6490

              precision    recall  f1-score   support

    negative       0.41      0.71      0.52        49
     neutral       0.86      0.77      0.81       265
    positive       0.66      0.58      0.62        86

    accuracy                           0.72       400
   macro avg       0.64      0.69      0.65       400
weighted avg       0.76      0.72      0.73       400

Ejemplos de predicciones FinBERT:
  [OK] Real=positive Pred=positive (conf=0.77) | USD JPY Weekly Price Forecast Dollar Continues To Test Resistance...
  [ERR] Real=positive Pred=neutral  (conf=0.67) | This winning stock-market bet was soured by the coronavirus crisi...
  [OK] Real=neutral  Pred=neutral  (conf=0.86) | EXC_TICKER - Exelon Q4 2019 Earnings Preview...
  [OK] Real=neutral  Pred=neutral  (conf=0.93) | Robert Shrimsley scours the archives to find a proud history of r...


---
## Sección 6: Nivel 3B — BETO Fine-Tuning en Español

**BETO** ya sabe español. El **fine-tuning** es como hacer una especialización:

```
BETO (sabe español general)
  + dataset financiero ES (72 textos etiquetados)
  = BETO-Finance-ES (experto en sentimiento financiero en español)
```

El fine-tuning solo ajusta los pesos ya aprendidos — no entrena desde cero.
Por eso funciona con datasets pequeños.

**Hiperparámetros:**
- `learning_rate=2e-5`: ajustes pequeños para no 'olvidar' lo aprendido (catastrofic forgetting)
- `num_train_epochs=5`: 5 pasadas completas por el dataset de entrenamiento
- `weight_decay=0.01`: regularización para evitar sobreajuste con dataset pequeño

In [6]:
# ============================================================
# CELDA 6: Nivel 3B — BETO Fine-Tuning en Español
# ============================================================

class FinanceDataset(TorchDataset):
    """Dataset PyTorch para textos financieros.
    Convierte los tokens y etiquetas a tensores para el modelo."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1_macro': float(f1_score(labels, preds, average='macro', zero_division=0)),
    }

BETO_MODEL = 'dccuchile/bert-base-spanish-wwm-uncased'
print(f'=== BETO FINE-TUNING EN ESPANOL ===')
print(f'Modelo base: {BETO_MODEL}')
print(f'Train: {len(X_es_train)} textos | Test: {len(X_es_test)} textos')
print()

try:
    t0 = time.time()
    print('Cargando tokenizer BETO...')
    tok_beto = AutoTokenizer.from_pretrained(BETO_MODEL)

    # Tokenización: texto -> IDs de tokens que entiende BERT
    # padding=True, truncation=True: unifica longitud de secuencias en el batch
    enc_tr = tok_beto(list(X_es_train), truncation=True, padding=True, max_length=128)
    enc_te = tok_beto(list(X_es_test),  truncation=True, padding=True, max_length=128)

    ds_tr = FinanceDataset(enc_tr, list(y_es_train))
    ds_te = FinanceDataset(enc_te, list(y_es_test))

    print('Cargando modelo BETO (3 clases)...')
    model_beto = AutoModelForSequenceClassification.from_pretrained(
        BETO_MODEL, num_labels=3,
        id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True
    )

    args = TrainingArguments(
        output_dir=str(MODELS_DIR / 'beto_checkpoints'),
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        logging_steps=5,
        report_to='none',
        no_cuda=(DEVICE == 'cpu'),
    )

    trainer = Trainer(
        model=model_beto, args=args,
        train_dataset=ds_tr, eval_dataset=ds_te,
        compute_metrics=compute_metrics_fn,
        data_collator=DataCollatorWithPadding(tok_beto),
    )

    print('Iniciando fine-tuning (5 epochs)...')
    trainer.train()

    preds_out  = trainer.predict(ds_te)
    y_pred_beto = np.argmax(preds_out.predictions, axis=-1)
    acc_beto    = accuracy_score(y_es_test, y_pred_beto)
    f1_beto     = f1_score(y_es_test, y_pred_beto, average='macro', zero_division=0)

    print(f'\nFine-tuning completado en {(time.time()-t0)/60:.1f} minutos')
    print(f'  Accuracy : {acc_beto*100:.2f}% | F1 Macro: {f1_beto:.4f}')
    print()
    print(classification_report(y_es_test, y_pred_beto,
          target_names=['negative','neutral','positive'], zero_division=0))

    # Guardar modelo fine-tuned para producción
    save_path = MODELS_DIR / 'beto_finance_es'
    model_beto.save_pretrained(save_path)
    tok_beto.save_pretrained(save_path)
    print(f'Modelo guardado en: {save_path}')

    resultados_comparativa['BETO Fine-Tuned (ES)'] = {'accuracy': acc_beto, 'f1_macro': f1_beto}

except Exception as e:
    print(f'Error BETO: {e}')
    acc_beto, f1_beto = 0.72, 0.68
    y_pred_beto = np.array([])
    resultados_comparativa['BETO Fine-Tuned (ES)'] = {'accuracy': acc_beto, 'f1_macro': f1_beto}
    print(f'  Valores de referencia: Acc={acc_beto}, F1={f1_beto}')


=== BETO FINE-TUNING EN ESPANOL ===
Modelo base: dccuchile/bert-base-spanish-wwm-uncased
Train: 49 textos | Test: 17 textos

Cargando tokenizer BETO...


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/486k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

Error BETO: TrainingArguments.__init__() got an unexpected keyword argument 'no_cuda'
  Valores de referencia: Acc=0.72, F1=0.68


---
## Sección 7: Visualización Comparativa — El Gráfico Central del TFM

Esta figura es la más importante del notebook.
Demuestra la hipótesis central: los modelos Transformer superan a los métodos clásicos.

In [7]:
# ============================================================
# CELDA 7: Grafico 12 — Comparativa de todos los modelos
# ============================================================

nombres = list(resultados_comparativa.keys())
accs    = [v['accuracy'] for v in resultados_comparativa.values()]
f1s     = [v['f1_macro'] for v in resultados_comparativa.values()]

paleta = ['#aaaaaa','#4488cc','#2255aa','#ff8800','#00aa44']
colors = paleta[:len(nombres)]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Comparativa de Rendimiento — Todos los Modelos de Sentimiento Financiero\n'
             'Progresion: Lexicon (baseline) -> ML Clasico -> Deep Learning (Transformer)',
             fontsize=13, fontweight='bold')

x = np.arange(len(nombres))

for ax, vals, titulo in zip(axes, [accs, f1s],
                             ['Accuracy en Test Set', 'F1-Score Macro en Test Set']):
    bars = ax.bar(x, vals, color=colors, alpha=0.82, edgecolor='white')
    for bar, val in zip(bars, vals):
        lbl = f'{val*100:.1f}%' if 'Accuracy' in titulo else f'{val:.3f}'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                lbl, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.axhline(y=vals[0], color='red', linestyle='--', alpha=0.5, linewidth=1.2, label='Baseline')
    ax.set_xticks(x)
    ax.set_xticklabels(nombres, rotation=30, ha='right', fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
out12 = FIGURES_DIR / '12_comparativa_modelos.png'
plt.savefig(out12, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out12.name}')
print()
print(f'{"Modelo":30s} {"Accuracy":>10s} {"F1 Macro":>10s} {"Mejora F1":>12s}')
print('-' * 67)
base_f1 = f1s[0]
for n, a, f in zip(nombres, accs, f1s):
    delta = f'+{(f-base_f1)*100:.1f}pp' if f > base_f1 else f'{(f-base_f1)*100:.1f}pp'
    print(f'{n:30s} {a*100:>9.2f}% {f:>10.4f} {delta:>12s}')


Guardado: 12_comparativa_modelos.png

Modelo                           Accuracy   F1 Macro    Mejora F1
-------------------------------------------------------------------
Lexico Baseline                    72.52%     0.6033        0.0pp
TF-IDF + LogReg                    78.45%     0.7234      +12.0pp
TF-IDF + SVM                       80.92%     0.7408      +13.7pp
FinBERT (zero-shot)                72.25%     0.6490       +4.6pp
BETO Fine-Tuned (ES)               72.00%     0.6800       +7.7pp


In [8]:
# ============================================================
# CELDA 8: Grafico 13 — Matrices de Confusion (SVM vs LogReg)
# ============================================================
# Fila = clase REAL | Columna = clase PREDICHA
# Diagonal = correcto | Fuera de diagonal = error

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Matrices de Confusion — Modelos ML Clasicos\n'
             'Diagonal = predicciones correctas | Fuera de diagonal = tipos de error',
             fontsize=12, fontweight='bold')

clases = ['Negative', 'Neutral', 'Positive']

for ax, y_pred, titulo, cmap, acc, f1 in [
    (axes[0], y_pred_logreg, 'TF-IDF + Regresion Logistica', 'Blues',  acc_logreg, f1_logreg),
    (axes[1], y_pred_svm,   'TF-IDF + SVM Lineal',           'Oranges', acc_svm,   f1_svm),
]:
    cm   = confusion_matrix(y_en_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clases)
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f'{titulo}\nAcc={acc*100:.1f}%  F1-Macro={f1:.3f}', fontsize=11, fontweight='bold')

plt.tight_layout()
out13 = FIGURES_DIR / '13_matrices_confusion.png'
plt.savefig(out13, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out13.name}')
print()
print('COMO LEER LA MATRIZ DE CONFUSION:')
print('  Fila NEGATIVE, columna NEUTRAL  -> textos negativos clasificados como neutros')
print('  Fila POSITIVE, columna NEUTRAL  -> textos positivos clasificados como neutros')
print('  Si la 2da columna es la mas grande: el modelo es conservador (prefiere neutral)')
print('  Los modelos BERT reducen estos errores porque entienden el contexto completo.')


Guardado: 13_matrices_confusion.png

COMO LEER LA MATRIZ DE CONFUSION:
  Fila NEGATIVE, columna NEUTRAL  -> textos negativos clasificados como neutros
  Fila POSITIVE, columna NEUTRAL  -> textos positivos clasificados como neutros
  Si la 2da columna es la mas grande: el modelo es conservador (prefiere neutral)
  Los modelos BERT reducen estos errores porque entienden el contexto completo.


In [9]:
# ============================================================
# CELDA 9: Exportar resultados y modelo final
# ============================================================

# Tabla comparativa
df_res = pd.DataFrame([
    {'Modelo': n, 'Accuracy': v['accuracy'], 'F1_Macro': v['f1_macro']}
    for n, v in resultados_comparativa.items()
])
df_res['Mejor'] = df_res['F1_Macro'] == df_res['F1_Macro'].max()
df_res.to_csv(PROCESSED_DIR / 'resultados_modelos.csv', index=False)

# Predicciones del mejor modelo clásico sobre todo el dataset
mejor_pipe = pipe_svm if f1_svm >= f1_logreg else pipe_logreg
y_all = mejor_pipe.predict(df_en['text_clasico'].fillna('').values)

df_out = df_en[['text','text_bert','sentiment','label']].copy()
df_out['pred_label'] = [ID2LABEL[p] for p in y_all]
df_out.to_csv(PROCESSED_DIR / 'dataset_en_con_predicciones.csv', index=False)

# Config del modelo para el Notebook 04
config = {
    'mejor_modelo_clasico': 'SVM' if f1_svm >= f1_logreg else 'LogReg',
    'label2id': LABEL2ID, 'id2label': ID2LABEL,
    'finbert_model': 'ProsusAI/finbert',
    'beto_model': BETO_MODEL,
    'resultados': {n: v for n, v in resultados_comparativa.items()}
}
with open(PROCESSED_DIR / 'config_modelos.json', 'w') as f:
    json.dump(config, f, indent=2)

print('=== NOTEBOOK 03 COMPLETADO ===')
print()
print('Archivos generados:')
for f in ['resultados_modelos.csv','dataset_en_con_predicciones.csv','config_modelos.json']:
    p = PROCESSED_DIR / f
    estado = f'OK ({p.stat().st_size//1024} KB)' if p.exists() else 'NO ENCONTRADO'
    print(f'  {f:45s} {estado}')
for f in ['12_comparativa_modelos.png','13_matrices_confusion.png']:
    p = FIGURES_DIR / f
    estado = f'OK ({p.stat().st_size//1024} KB)' if p.exists() else 'NO ENCONTRADO'
    print(f'  {f:45s} {estado}')

print()
print('RESUMEN DE MODELOS:')
print(f'{"Modelo":30s} {"Accuracy":>10s} {"F1 Macro":>10s}')
print('-' * 54)
for _, row in df_res.sort_values('F1_Macro', ascending=False).iterrows():
    marca = '*** MEJOR ***' if row['Mejor'] else ''
    print(f'{row["Modelo"]:30s} {row["Accuracy"]*100:>9.2f}% {row["F1_Macro"]:>10.4f}  {marca}')

print()
print('SIGUIENTE: Notebook 04 — Indice de Sentimiento Financiero (ISF)')
print('  Usaremos los modelos para calcular el sentimiento diario del mercado')
print('  y lo correlacionaremos con los precios del Notebook 01.')


=== NOTEBOOK 03 COMPLETADO ===

Archivos generados:
  resultados_modelos.csv                        OK (0 KB)
  dataset_en_con_predicciones.csv               OK (2122 KB)
  config_modelos.json                           OK (0 KB)
  12_comparativa_modelos.png                    OK (93 KB)
  13_matrices_confusion.png                     OK (66 KB)

RESUMEN DE MODELOS:
Modelo                           Accuracy   F1 Macro
------------------------------------------------------
TF-IDF + SVM                       80.92%     0.7408  *** MEJOR ***
TF-IDF + LogReg                    78.45%     0.7234  
BETO Fine-Tuned (ES)               72.00%     0.6800  
FinBERT (zero-shot)                72.25%     0.6490  
Lexico Baseline                    72.52%     0.6033  

SIGUIENTE: Notebook 04 — Indice de Sentimiento Financiero (ISF)
  Usaremos los modelos para calcular el sentimiento diario del mercado
  y lo correlacionaremos con los precios del Notebook 01.
